# GFMC tests

Run this notebook from the repository root or from `notebook/`. It builds every lattice with `ising`, then calls `GFMC` with the transverse field `gamma` and the longitudinal field `h`.

The energy cells run the projector. Expect about a minute. A row passes when the autocorrelation-aware error is marked resolved, is at most `0.03`, and both the mixed and growth energies lie within `max(0.03, 3 errors)` of the reference. One-spin references are analytic. The other references are exact diagonalization, which stops at 16 sites.


In [1]:
from pathlib import Path
import math
import sys

import numpy as np

root = Path.cwd()
if not (root / "code").exists():
    root = root.parent
sys.path.insert(0, str(root / "code"))

import gfmc
from gfmc import ConstantGuide, GFMC
from ising import chain, square
from kagome_qea.exact import ground_state

ATOL = 0.03
SETTINGS = dict(n_walkers=2500, tau_block=0.2, n_equilibrate=40, n_measure=160)


## The projector does not build the lattice

`chain` and `square` live in `ising`. `h` on those constructors is stored on the operator. Passing `h` to `GFMC` replaces that value and leaves the original bonds alone.


In [ ]:
assert not hasattr(gfmc, "chain")
assert not hasattr(gfmc, "square")

open_chain = chain(8, J=1.0, periodic=False)
periodic_chain = chain(8, J=1.0, periodic=True)
patch = square(3, 2, J=1.0, periodic=False)
assert open_chain.n_sites == 8 and open_chain.bonds.shape == (7, 2)
assert periodic_chain.bonds.shape == (8, 2)
assert patch.n_sites == 6 and patch.bonds.shape == (7, 2)

bare = chain(1)
assert bare.h == 0.0
kept = GFMC(chain(1, h=0.4), gamma=1.0)
assert kept.h == 0.4
replaced = GFMC(bare, gamma=0.8, h=0.3)
assert replaced.h == 0.3 and bare.h == 0.0
assert ConstantGuide().log_ratio(np.array([1]), 0) == 0.0

try:
    square(1, 4, periodic=True)
    raise AssertionError("a periodic square of side 1 was accepted")
except ValueError:
    pass

try:
    GFMC(object(), gamma=1.0)
    raise AssertionError("an object without the operator methods was accepted")
except TypeError:
    pass

print("operator interface passed")


## Energies

Each call compares the mixed estimator and the growth estimator with the reference. `h` is passed to `GFMC` for the longitudinal-field rows. The reference operator is the same bond list with that field.


In [ ]:
rows = []

def check(name, hamiltonian, gamma, exact=None, h=None, configurations=None, seed=0):
    reference = hamiltonian if h is None else hamiltonian.with_field(h)
    if exact is None:
        exact = ground_state(reference, gamma, moments=False)["energy"]
    result = GFMC(hamiltonian, gamma, h=h).run(
        seed=seed, configurations=configurations, **SETTINGS
    )
    assert result["h"] == reference.h
    resolved = result["energy_resolved"] and result["energy_growth_resolved"]
    for label, got, stderr in (
        ("mixed", result["energy"], result["energy_error"]),
        ("growth", result["energy_growth"], result["energy_growth_error"]),
    ):
        difference = abs(got - exact)
        ok = resolved and math.isfinite(stderr) and stderr <= ATOL and difference <= max(ATOL, 3.0 * stderr)
        status = "pass" if ok else "FAIL"
        print(
            f"{status}  {name} {label}: {got:.4f} ± {stderr:.4f}  "
            f"exact {exact:.4f}  |diff| {difference:.4f}"
        )
        rows.append(ok)

check("one spin", chain(1), gamma=1.0, exact=-1.0, configurations=np.array([1]), seed=1)
check(
    "one spin",
    chain(1),
    gamma=0.8,
    h=0.3,
    exact=-math.sqrt(0.3 ** 2 + 0.8 ** 2),
    configurations=np.array([1]),
    seed=2,
)
check("open chain N=8", chain(8), gamma=1.0, seed=3)
check("periodic chain N=8", chain(8, periodic=True), gamma=1.0, seed=4)
check("open chain N=6", chain(6), gamma=1.0, h=0.25, seed=5)
check("open square 3x2", square(3, 2), gamma=1.0, h=0.2, seed=6)

if not all(rows):
    raise AssertionError(f"{rows.count(False)} energy comparisons failed")
print(f"passed {len(rows)} energy comparisons")
